In [1]:
!pip install -q trl transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.3 MB/s eta 0:00:00


In [2]:
!pip install -q datasets

In [3]:
!pip install -q --upgrade wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 68.0 MB/s eta 0:00:00


In [4]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

In [5]:
# Model from Hugging Face hub
base_model = "meta-llama/Llama-3.2-3B-Instruct"

# New instruction dataset
med_dataset = "/content/medquad-guanco-llama2"

# Fine-tuned model
new_model = "med_chatbot"

In [6]:
# Load the dataset directly from Hugging Face hub
dataset = load_dataset('keivalya/MedQuad-MedicalQnADataset', split='train')
print('Dataset loaded successfully!')
print(dataset[0])

README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

Dataset loaded successfully!
{'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?', 'Answer': 'LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'}


In [7]:
len(dataset)

16407

In [8]:
compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

In [9]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map="auto",     # {"": 0}
    torch_dtype=torch.float16
)
model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [11]:
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [12]:
peft_params = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [13]:
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=1e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    #group_by_length=True,
    lr_scheduler_type="constant",
    report_to="tensorboard"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
# reate the Llama 3.2 chat template format
def format_llama3(example):
    system_prompt = "You are a helpful and accurate medical assistant."
    user_msg = example["Question"]
    model_msg = example["Answer"]

    # Llama 3.2 specific token structure
    text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_msg}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{model_msg}<|eot_id|>"

    example["text"] = text
    return example

dataset = dataset.map(format_llama3)

dataset = dataset.filter(lambda x: x["text"] is not None and isinstance(x["text"], str))

Map:   0%|          | 0/16407 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16407 [00:00<?, ? examples/s]

In [15]:
len(dataset)

16407

In [16]:
#for 1000 rows only

"""
#Shuffle the data and select a smaller chunk (eg: 1000 rows)
num_samples = 1000 
dataset = dataset.shuffle(seed=42).select(range(num_samples))

print(f"Training on a subset of {len(dataset)} examples.")"""

'\n#Shuffle the data and select a smaller chunk (eg: 1000 rows)\nnum_samples = 1000 \ndataset = dataset.shuffle(seed=42).select(range(num_samples))\n\nprint(f"Training on a subset of {len(dataset)} examples.")'

In [17]:
use_fp16 = False
use_bf16 = False

# Define trainer arguments
training_params = SFTConfig(
    output_dir="./results",
    dataset_text_field="text",
    max_length=512,
    packing=False,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    save_steps=500,
    save_total_limit=2,
    logging_steps=25,
    fp16=use_fp16,
    bf16=use_bf16,
    max_grad_norm=0.0,
    report_to="tensorboard",
)

# Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_params,
    args=training_params,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/16407 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16407 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16407 [00:00<?, ? examples/s]

2026-03-21 08:27:25.102885: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774081645.471709      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774081645.572101      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774081646.493601      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774081646.493638      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774081646.493641      23 computation_placer.cc:177] computation placer alr

In [18]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
25,2.461036
50,2.463355
75,2.293358
100,2.282044
125,2.213114
150,2.039291
175,1.892741
200,1.848541
225,1.787796
250,1.662490


TrainOutput(global_step=16407, training_loss=1.37341947419527, metrics={'train_runtime': 10225.5826, 'train_samples_per_second': 1.605, 'train_steps_per_second': 1.605, 'total_flos': 7.236998738061312e+16, 'train_loss': 1.37341947419527})

In [19]:
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

('med_chatbot/tokenizer_config.json',
 'med_chatbot/chat_template.jinja',
 'med_chatbot/tokenizer.json')

In [20]:
from tensorboard import notebook
log_dir = "results/runs"
notebook.start("--logdir {} --port 4000".format(log_dir))

<IPython.core.display.Javascript object>

In [21]:
logging.set_verbosity(logging.CRITICAL)
model.eval()

prompt = "What are the common symptoms of back pain?"
system_prompt = "You are a helpful and accurate medical assistant."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt},
]
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
 )


eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
stop_ids = [tokenizer.eos_token_id]
if eot_id is not None and eot_id != tokenizer.eos_token_id:
    stop_ids.append(eot_id)

pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
 )

result = pipe(
    prompt_text,
    max_new_tokens=120,
    do_sample=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=4,
    eos_token_id=stop_ids,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False,
 )

answer_only = result[0]["generated_text"].strip()
print(answer_only)

Back pain is a very common symptom with many possible causes. The following are some common symptoms associated with back pain: 

- Pain in your back 
- Pain that radiates to your legs 
- Muscle weakness 
- Numbness or tingling in your legs 
 - Pain when you move 
 - Pain that worsens at night 
 - Pain in your lower back (lumbar region) 
 - Pain above your waist (thoracic region) 
- Pain below your waist (cervical region) 
   - Pain in one area of your back 
   - Back pain that comes


In [22]:
model,tokenizer

(LlamaForCausalLM(
   (model): LlamaModel(
     (embed_tokens): Embedding(128256, 3072)
     (layers): ModuleList(
       (0-27): 28 x LlamaDecoderLayer(
         (self_attn): LlamaAttention(
           (q_proj): lora.Linear4bit(
             (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
             (lora_dropout): ModuleDict(
               (default): Dropout(p=0.1, inplace=False)
             )
             (lora_A): ModuleDict(
               (default): Linear(in_features=3072, out_features=64, bias=False)
             )
             (lora_B): ModuleDict(
               (default): Linear(in_features=64, out_features=3072, bias=False)
             )
             (lora_embedding_A): ParameterDict()
             (lora_embedding_B): ParameterDict()
             (lora_magnitude_vector): ModuleDict()
           )
           (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
           (v_proj): lora.Linear4bit(
             (base_layer): Lin

In [23]:
model.push_to_hub('Lucifer049/med_chatbot_finetuned', create_pr=True)
tokenizer.push_to_hub('Lucifer049/med_chatbot_finetuned', create_pr=True)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Lucifer049/med_chatbot_finetuned/commit/c6825521bfdd4c5055a2229538a052e6ad37e920', commit_message='Upload tokenizer', commit_description='', oid='c6825521bfdd4c5055a2229538a052e6ad37e920', pr_url='https://huggingface.co/Lucifer049/med_chatbot_finetuned/discussions/2', repo_url=RepoUrl('https://huggingface.co/Lucifer049/med_chatbot_finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Lucifer049/med_chatbot_finetuned'), pr_revision='refs/pr/2', pr_num=2)

In [24]:
load_model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

model = PeftModel.from_pretrained(load_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [25]:
load_model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

model = PeftModel.from_pretrained(load_model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it safely
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
# Only use existing tokens to prevent embedding size mismatches
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [26]:
model.push_to_hub(new_model)
tokenizer.push_to_hub(new_model)

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Lucifer049/med_chatbot/commit/f8e44b39eccb929f8147b434bd9e742719c8bf77', commit_message='Upload tokenizer', commit_description='', oid='f8e44b39eccb929f8147b434bd9e742719c8bf77', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Lucifer049/med_chatbot', endpoint='https://huggingface.co', repo_type='model', repo_id='Lucifer049/med_chatbot'), pr_revision=None, pr_num=None)